# Hackathon · Team 4 · Model v3: the same network as v2, trained for what the app needs (train on Colab)

**Why a v3.** v2 is better than v1, but with **the new step-3 rules** (1.6 m between parallel walls, 4 or 6 corners, bedroom at least 9 m², bathroom doors) only about **22% of v2's pictures are usable** (40 validation requests, 10 pictures each, checked with the latest `layout_core.py`). The main reasons a picture is dropped:

| reason | share of all pictures |
|---|---|
| a living room part is narrower than 1.6 m | 11% |
| a bedroom is too small | 10% |
| a bedroom has too many corners | 10% |
| a kitchen part is narrower than 1.6 m | 8% |

These are about **the shapes** the model draws, not about the request. v2 chose its best epoch by "bedrooms right + balcony right" in the raw pictures, so it never saw these rules. v3 learns with the rules in view.

| | v2 | v3 |
|---|---|---|
| the network | `LayoutCVAE2` (FiLM) | **the same** (the server and `publish.py` work without any change) |
| which epoch is kept | the best "bedrooms right + balcony right" in the raw pictures | the epoch whose pictures **pass step 3 most often** (30 validation requests × 10 pictures, the latest rules) |
| Dice (overlap) in the loss | all 8 layers the same | small rooms weigh more: kitchen ×3, bathroom ×2, bedroom and balcony ×1.5 |
| rule loss (new) | none | a fine-tuning at the end draws plans from **random numbers** (as the app does) and pushes the decoder when the bedrooms are too small together, or the kitchen or the bathroom is missing. It is kept only if step 3 says it helps |
| training | up to 120 epochs, stop after 15 without a better score | up to 150 epochs, stop after 30 |

**The bedroom checker of the first v2 run is not used again.** It was a second network that had to learn to count bedrooms, it learned badly (48% right) and broke the model. The rule loss has nothing to learn: it measures room areas directly from the drawn map.

At the end, **v1, v2 and v3 are compared on the test plans** (plans no model saw and no choice was made on), through step 3 with the latest rules. The app gets the winner.

### Before you run it (5 minutes)
1. In Google Drive, open the folder **`team4_v2`** (from v2). It already has the data and v1. Add or replace:
   - `layout_core.py`: **replace it** with the latest one from `01_autoencoder/notebook/` (the new rules are in it)
   - `layout_cvae_v2.pt`: from `01_autoencoder/models/layout_generator_v2/` (for the comparison)
2. Open this notebook in Colab: **File → Upload notebook**.
3. **Runtime → Change runtime type → T4 GPU**.
4. **Runtime → Run all**. Allow access to your Drive.

**Time:** about 1-1.5 hours on a T4 (most of it is step 3, which runs on the CPU). Everything is saved in `team4_v2/v3/` **after every epoch**: if Colab disconnects, run all again and it continues. That works only with Drive; without Drive (the upload fallback) a disconnection starts it again.

# 0. Setup

In [ ]:
import json
import os
import shutil
import sys
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F

IN_COLAB = "google.colab" in sys.modules
FILES = ["generator_maps_128px.npz", "generator_lib.csv", "layout_cvae.pt", "layout_cvae_config.json",
         "room_rules.json", "layout_core.py", "layout_cvae_v2.pt"]
if IN_COLAB:
    from google.colab import drive, files
    WORK = None
    try:                                                         # 1st choice: the team4_v2 folder in Google Drive
        drive.mount("/content/drive")
        if Path("/content/drive/MyDrive/team4_v2").exists():
            WORK = Path("/content/drive/MyDrive/team4_v2")
        else:
            print("Google Drive is connected, but there is no team4_v2 folder in My Drive")
    except Exception as e:
        print(f"Google Drive could not be connected ({e}).")
    if WORK is None:                                             # 2nd choice: upload the 7 files into this session
        WORK = Path("/content/team4_v2")
        WORK.mkdir(exist_ok=True)
        if not all((WORK / f).exists() for f in FILES):
            print("Choose these 7 files on your computer (select all together):", ", ".join(FILES))
            for name, data in files.upload().items():
                (WORK / name).write_bytes(data)
        print("Working without Drive: if Colab disconnects, the training starts again. At the end the results are downloaded as a zip.")
    DATA_DIR = MODEL_V1_DIR = CODE_DIR = WORK
    V2_WEIGHTS = next((p for p in [WORK / "layout_cvae_v2.pt", WORK / "v2b" / "layout_cvae_v2.pt"] if p.exists()),
                      WORK / "layout_cvae_v2.pt")
    OUT_DIR = WORK / "v3"
    os.system("pip install -q onnx onnxruntime")
else:                                                            # on your computer, from 01_autoencoder/notebook
    DATA_DIR = Path("../../data")
    MODEL_V1_DIR = Path("../models") / "layout_generator"
    CODE_DIR = Path(".")
    V2_WEIGHTS = Path("../models") / "layout_generator_v2" / "layout_cvae_v2.pt"
    OUT_DIR = Path("../models") / "layout_generator_v3"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUT_DIR if IN_COLAB else Path("../figures")
needed = [DATA_DIR / "generator_maps_128px.npz", DATA_DIR / "generator_lib.csv", MODEL_V1_DIR / "layout_cvae.pt",
          MODEL_V1_DIR / "layout_cvae_config.json", MODEL_V1_DIR / "room_rules.json", CODE_DIR / "layout_core.py", V2_WEIGHTS]
missing = [str(p) for p in needed if not p.exists()]
assert not missing, f"missing files: {missing}"

sys.path.insert(0, str(CODE_DIR))
import layout_core as lc
assert hasattr(lc, "LayoutCVAE2"), "layout_core.py is too old: copy the latest one"
assert "min_clear_width_m" in lc.SETTINGS and hasattr(lc, "_bad_passage"), \
    "layout_core.py does not have the new rules (1.6 m, corners): copy the latest one from 01_autoencoder/notebook"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    torch.set_num_threads(os.cpu_count() or 4)
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
print(f"PyTorch {torch.__version__} | device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else
      "  <- no GPU: in Colab, Runtime > Change runtime type > T4 GPU"))


def save_fig(name):
    path = FIG_DIR / f"{name}.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print(f"Figure saved to {path}")

# 1. The data (the same as v1 and v2)

15,832 plans, 8 layers of 128 × 128, the same split: train 12,682, validation 1,580, test 1,570. New here: each training plan also carries its **scale** (metres per pixel) and its **area**, because the rule loss measures rooms in m².

In [ ]:
cache = np.load(DATA_DIR / "generator_maps_128px.npz")
sil_u8, maps_u8 = cache["sil"], cache["maps"]                       # read once (each read of the npz unpacks it again)
scales, sil_units2 = cache["scales"], cache["sil_units2"]
lib = pd.read_csv(DATA_DIR / "generator_lib.csv")
assert np.array_equal(lib["index"].to_numpy(), cache["index"]), "generator_lib.csv does not match the room maps"
lib["area_m2"] = sil_units2 * lib["m_per_unit"] ** 2
lib["m_per_px"] = lib["m_per_unit"] * (512 / lc.IMG) / scales

config_v1 = json.loads((MODEL_V1_DIR / "layout_cvae_config.json").read_text())
AREA_MEAN, AREA_STD = config_v1["condition"]["area_mean"], config_v1["condition"]["area_std"]
rules = json.loads((MODEL_V1_DIR / "room_rules.json").read_text())
conds = np.stack([lc.condition_vector(b, y, a, AREA_MEAN, AREA_STD)
                  for b, y, a in zip(lib["bedrooms"], lib["balcony"], lib["area_m2"])])
bed_class = np.clip(lib["bedrooms"].to_numpy(), 1, lc.BED_MAX) - 1          # 0..4 = 1..5+ bedrooms

split = lib["split"].to_numpy()
train_rows, val_rows, test_rows = (np.where(split == name)[0] for name in ["train", "val", "test"])
print(f"{len(lib):,} plans | train {len(train_rows):,} | validation {len(val_rows):,} | test {len(test_rows):,}")

store = "cuda" if DEVICE == "cuda" else "cpu"                       # on the GPU the training plans stay as bytes
X_train = torch.from_numpy(maps_u8[train_rows]).to(store)
S_train = torch.from_numpy(sil_u8[train_rows]).unsqueeze(1).to(store)
C_train = torch.from_numpy(conds[train_rows]).float().to(store)
B_train = torch.from_numpy(bed_class[train_rows]).long().to(store)
M_train = torch.from_numpy(lib["m_per_px"].to_numpy()[train_rows]).float().to(store)
A_train = torch.from_numpy(lib["area_m2"].to_numpy()[train_rows]).float().to(store)
val_set = tuple(torch.from_numpy(a) for a in (maps_u8[val_rows], sil_u8[val_rows][:, None], conds[val_rows]))

rng = np.random.default_rng(0)
score_rows = rng.choice(val_rows, size=100, replace=False)                 # raw pictures: bedrooms / balcony / different
select_rows = rng.choice(np.setdiff1d(val_rows, score_rows), size=30, replace=False)   # step 3: which epoch is kept
SELECT_PICTURES = 10


def batch(idx):
    # one training batch, 0-1, with a random mirror (the same for maps and silhouette)
    x = X_train[idx].to(DEVICE).float() / 255
    s = S_train[idx].to(DEVICE).float() / 255
    if torch.rand(1).item() < 0.5:
        x, s = x.flip(3), s.flip(3)
    if torch.rand(1).item() < 0.5:
        x, s = x.flip(2), s.flip(2)
    return x, s, C_train[idx].to(DEVICE), B_train[idx].to(DEVICE), M_train[idx].to(DEVICE), A_train[idx].to(DEVICE)

# 2. The loss: the v2 loss with heavier small rooms, and the rule loss

**The v2 loss** (BCE + Dice + β·KL with free bits 0.5, so the 5 options stay different). New: in the **Dice** part (how well each room overlaps the real one), the kitchen counts 3 times, the bathroom twice, bedrooms and balcony 1.5 times. Without it, a missing kitchen costs as much as a slightly wrong living room.

**The rule loss** works on plans drawn from **random numbers**, exactly like the app. It measures in m² how much area each room type gets and pushes only when a plan is clearly too poor:
- all bedrooms together: at least **12 m² × the bedrooms asked** (in the data, 95% of the plans have more; it includes the walls, so about the 9 m² net of step 3), never more than half of the flat
- kitchen: at least **5 m²**; bathrooms together: at least **4.5 m²** (95% of the real plans have more)

A plan that already has enough area costs nothing. The area is "soft" (a pixel counts by how sure the decoder is), so the push can reach the decoder.

In [ ]:
#                          living bedroom bathroom kitchen balcony wall door window
CH_WEIGHTS = torch.tensor([1.0,   1.0,    2.0,     3.0,    2.0,    3.0, 8.0, 8.0], device=DEVICE).view(1, -1, 1, 1)
DICE_WEIGHTS = torch.tensor([1.0, 1.5,    2.0,     3.0,    1.5,    1.0, 1.0, 1.0], device=DEVICE)
DICE_WEIGHTS = (DICE_WEIGHTS / DICE_WEIGHTS.mean()).view(1, -1)      # the same total size as v2's Dice
FREE_BITS = 0.5
BEDROOM_M2, KITCHEN_M2, BATHROOM_M2 = 12.0, 5.0, 4.5


def cvae_loss(out, x, mu, logvar, beta):
    bce = (F.binary_cross_entropy(out, x, reduction="none") * CH_WEIGHTS).mean()
    inter = (out * x).sum(dim=(2, 3))
    total = out.sum(dim=(2, 3)) + x.sum(dim=(2, 3))
    dice = ((1 - (2 * inter + 1) / (total + 1)) * DICE_WEIGHTS).mean()
    kl_per_number = (-0.5 * (1 + logvar - mu ** 2 - logvar.exp())).mean(0)
    kl_loss = kl_per_number.clamp(min=FREE_BITS).sum()
    return bce + dice + beta * kl_loss, bce.item(), dice.item(), kl_per_number.sum().item()


def room_m2(maps, m_per_px, sharp=10.0):
    # soft area of each room type in m2: (batch, 8, H, W) -> (batch, 5). A pixel goes to the strongest room
    # (softmax), weighted by how strong it is: faint colour spread over the plan counts almost nothing
    rooms = maps[:, :len(lc.ROOMS)]
    own = torch.softmax(rooms * sharp, dim=1) * rooms.max(dim=1, keepdim=True).values
    return own.sum(dim=(2, 3)) * m_per_px.view(-1, 1) ** 2


def rule_loss(maps, bed_cls, m_per_px, area_m2):
    a = room_m2(maps, m_per_px)
    i_bed, i_bath, i_kit = lc.ROOMS.index("bedroom"), lc.ROOMS.index("bathroom"), lc.ROOMS.index("kitchen")
    want_bed = torch.minimum((bed_cls + 1).float() * BEDROOM_M2, 0.5 * area_m2)
    want_kit = torch.minimum(torch.full_like(area_m2, KITCHEN_M2), 0.08 * area_m2)
    want_bath = torch.minimum(torch.full_like(area_m2, BATHROOM_M2), 0.06 * area_m2)
    short = (F.relu(want_bed - a[:, i_bed]) / want_bed + F.relu(want_kit - a[:, i_kit]) / want_kit
             + F.relu(want_bath - a[:, i_bath]) / want_bath)              # 0 when every room is big enough
    return short.mean()


# check on the real plans: real maps should cost (almost) nothing
x, s, c, beds, mpp, area = batch(torch.arange(256, device=store))
print(f"rule loss on 256 real plans: {rule_loss(x, beds, mpp, area).item():.3f} (0 = all rooms big enough)")
print(f"v2/v3 network: {sum(p.numel() for p in lc.LayoutCVAE2().parameters()):,} parameters")

# 3. Scoring

- **Raw pictures** (as in v2): 100 validation requests × 3 pictures: bedrooms right, balcony right, how different the options are.
- **Step 3** (new): 30 other validation requests × 10 pictures through `layout_core.design_from_maps` with the latest rules: the share of **usable pictures**, and the requests that get 5 options. The same random numbers every time, so two epochs are compared fairly.

**Score = share of usable pictures** (−1 if the options are almost the same: less than 30% of the pixels differ). Step 3 takes about a minute, so it runs every 3 epochs.

In [ ]:
def count_in_map(maps, m_per_px, min_bedroom_m2=6.0, min_balcony_m2=1.0):
    rooms = maps[:len(lc.ROOMS)]
    label, strong = rooms.argmax(0), rooms.max(0) > 0.5
    not_wall = maps[lc.CHANNELS.index("wall")] < 0.35
    bed = ((label == lc.ROOMS.index("bedroom")) & strong & not_wall).astype(np.uint8)
    _, _, stats, _ = cv2.connectedComponentsWithStats(bed, connectivity=4)
    bedrooms = int((stats[1:, cv2.CC_STAT_AREA] * m_per_px ** 2 >= min_bedroom_m2).sum())
    balcony_m2 = ((label == lc.ROOMS.index("balcony")) & strong).sum() * m_per_px ** 2
    return bedrooms, int(balcony_m2 >= min_balcony_m2)


def label_map(maps, sil):
    label = np.where(maps[:len(lc.ROOMS)].max(0) > 0.5, maps[:len(lc.ROOMS)].argmax(0), -1)
    return label[np.asarray(sil) > 127]


def raw_score(model, rows, n=3):
    beds_ok, balcony_ok, different = [], [], []
    for k, r in enumerate(rows):
        options = lc.generate(model, sil_u8[r], conds[r], n=n, seed=k, device=DEVICE)
        for o in options:
            beds, balcony = count_in_map(o, lib.loc[r, "m_per_px"])
            beds_ok.append(min(beds, lc.BED_MAX) == bed_class[r] + 1)
            balcony_ok.append(balcony == lib.loc[r, "balcony"])
        labels = [label_map(o, sil_u8[r]) for o in options]
        different.append(np.mean([(labels[i] != labels[j]).mean() for i in range(n) for j in range(i + 1, n)]))
    return {"bedrooms_right": float(np.mean(beds_ok)), "balcony_right": float(np.mean(balcony_ok)),
            "different": float(np.mean(different))}


def step3_score(model, rows, n=SELECT_PICTURES, seed=1000):
    # the pictures through step 3 (latest rules): usable share, requests with 5 options, why pictures are dropped
    usable, five, reasons = [], [], Counter()
    for k, r in enumerate(rows):
        maps = lc.generate(model, sil_u8[r], conds[r], n=n, seed=seed + k, device=DEVICE)
        options, why = lc.design_from_maps(maps, sil_u8[r], int(lib.loc[r, "bedrooms"]), int(lib.loc[r, "balcony"]),
                                           lib.loc[r, "m_per_px"], rules)
        usable.append(why.get("ok", 0) / n)
        five.append(len(options) == 5)
        reasons.update(why)
    return {"usable": float(np.mean(usable)), "five_options": float(np.mean(five)), "reasons": reasons}


def full_score(model):
    raw = raw_score(model, score_rows)
    s3 = step3_score(model, select_rows)
    score = s3["usable"] - (1.0 if raw["different"] < 0.3 else 0.0)
    return {**raw, "usable": s3["usable"], "five_options": s3["five_options"], "score": score}


@torch.no_grad()
def evaluate(model, data, batch_size=500):
    model.eval()
    sums, n = np.zeros(3), 0
    for i in range(0, len(data[0]), batch_size):
        x, s, c = (t[i:i + batch_size].to(DEVICE).float() for t in data)
        x, s = x / 255, s / 255
        mu, logvar = model.encode(x, s, c)
        _, bce, dice, kl = cvae_loss(model.decode(mu, s, c), x, mu, logvar, 0.0)
        sums += np.array([bce, dice, kl]) * len(x)
        n += len(x)
    return dict(zip(["val_bce", "val_dice", "val_kl"], sums / n))


v2 = lc.LayoutCVAE2().to(DEVICE)
v2.load_state_dict(torch.load(V2_WEIGHTS, map_location=DEVICE))
start = time.time()
V2_SELECT = full_score(v2)
print(f"v2 on the same check: usable {V2_SELECT['usable']:.0%} | 5 options {V2_SELECT['five_options']:.0%} | "
      f"bedrooms right {V2_SELECT['bedrooms_right']:.0%} | different {V2_SELECT['different']:.0%}  "
      f"({time.time() - start:.0f} s for one check)")

# 4. Training

1. **Main training**: v2's network from zero, the new Dice weights, up to `MAX_EPOCHS`. Every 3 epochs, the step-3 check; the best epoch is kept. It stops after `PATIENCE` epochs without a better score.
2. **Fine-tuning with the rule loss**: from the best epoch, 9 more epochs with a small learning rate, once with `RULE_WEIGHTS[0]` and once with `RULE_WEIGHTS[1]`. A fine-tuned model replaces the main one **only if its step-3 score is higher**: the rule loss cannot make v3 worse.

Everything is saved after each epoch in `OUT_DIR`: run the cell again and it continues.

In [ ]:
BETA = 0.008
MAX_EPOCHS, PATIENCE, WARMUP, EVAL_EVERY = 150, 30, 10, 3
BATCH, LR = 128, 1e-3
RULE_WEIGHTS = [0.1, 0.3]
FT_EPOCHS, FT_LR = 9, 2e-4
if DEVICE == "cpu":                  # no GPU (the laptop): a shorter plan
    MAX_EPOCHS, PATIENCE = 60, 15
WEIGHTS_V3 = OUT_DIR / "layout_cvae_v3.pt"
TRAIN_LOG = OUT_DIR / "layout_cvae_v3_training.json"


def train(model, epochs, lr, rule_weight, weights_path, checkpoint_path, patience=None, warmup=WARMUP, name=""):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr / 20)
    history, best, best_epoch, first = [], -np.inf, 0, 1
    if checkpoint_path.exists():                               # continue after a disconnection
        ck = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)     # our own file
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        history, best, best_epoch, first = ck["history"], ck["best"], ck["best_epoch"], ck["epoch"] + 1
        print(f"{name}: continuing after epoch {ck['epoch']} (best so far: epoch {best_epoch})")
    rows_t = torch.arange(len(train_rows), device=store)
    start = time.time()
    for epoch in range(first, epochs + 1):
        if patience and epoch - max(best_epoch, warmup) > patience:
            print(f"{name}: no better score for {patience} epochs, stopping")
            break
        b = BETA * min(1.0, epoch / warmup) if warmup else BETA
        model.train()
        total, total_rule = 0.0, 0.0
        for idx in rows_t[torch.randperm(len(rows_t), device=store)].split(BATCH):
            x, s, c, beds, mpp, area = batch(idx)
            out, mu, logvar = model(x, s, c)
            loss, *_ = cvae_loss(out, x, mu, logvar, b)
            if rule_weight > 0:                                 # draw from random numbers, as the app does
                drawn = model.decode(torch.randn_like(mu), s, c)
                rl = rule_loss(drawn, beds, mpp, area)
                loss = loss + rule_weight * rl
                total_rule += rl.item() * len(idx)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(idx)
        sched.step()
        row = {"epoch": epoch, "beta": b, "lr": sched.get_last_lr()[0], "train_loss": total / len(rows_t),
               "rule_loss": total_rule / len(rows_t), **evaluate(model, val_set)}
        line = f"{name} epoch {epoch:3d}  val BCE {row['val_bce']:.4f}  Dice {row['val_dice']:.3f}  KL {row['val_kl']:5.1f}"
        if epoch % EVAL_EVERY == 0 or epoch == epochs:
            row.update(full_score(model))
            line += (f"  |  usable {row['usable']:.0%}  5 options {row['five_options']:.0%}  bedrooms right "
                     f"{row['bedrooms_right']:.0%}  different {row['different']:.0%}")
            if epoch >= warmup and row["score"] > best:
                best, best_epoch = row["score"], epoch
                torch.save(model.state_dict(), weights_path)
                line += "  <- best"
        history.append({k: (int(v) if k == "epoch" else float(v)) for k, v in row.items()})
        print(line + f"  ({time.time() - start:.0f} s)")
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "history": history, "best": best, "best_epoch": best_epoch, "epoch": epoch}, checkpoint_path)
    model.load_state_dict(torch.load(weights_path, map_location=DEVICE))   # keep the best epoch
    return model, history, best_epoch, best

In [ ]:
RETRAIN = False          # False: if v3 is already trained (TRAIN_LOG exists), load it instead of training again

if TRAIN_LOG.exists() and WEIGHTS_V3.exists() and not RETRAIN:
    log = json.loads(TRAIN_LOG.read_text())
    history, best_epoch, finetune, RULE_WEIGHT = log["history"], log["best_epoch"], log["finetune"], log["rule_weight"]
    v3 = lc.LayoutCVAE2().to(DEVICE)
    v3.load_state_dict(torch.load(WEIGHTS_V3, map_location=DEVICE))
    print(f"loaded v3: best epoch {best_epoch} of {len(history)}, rule weight {RULE_WEIGHT}")
else:
    start = time.time()
    torch.manual_seed(RANDOM_STATE)
    base, history, best_epoch, base_score = train(lc.LayoutCVAE2().to(DEVICE), MAX_EPOCHS, LR, 0.0,
                                                  OUT_DIR / "main_best.pt", OUT_DIR / "main_checkpoint.pt",
                                                  patience=PATIENCE, name="main")
    print(f"\nmain training: best epoch {best_epoch}, score {base_score:.3f} ({(time.time() - start) / 60:.0f} minutes)\n")
    finetune = [{"rule_weight": 0.0, "score": float(base_score)}]
    for w in RULE_WEIGHTS:
        model = lc.LayoutCVAE2().to(DEVICE)
        model.load_state_dict(torch.load(OUT_DIR / "main_best.pt", map_location=DEVICE))
        _, h, e, s = train(model, FT_EPOCHS, FT_LR, w, OUT_DIR / f"rule_{w}_best.pt", OUT_DIR / f"rule_{w}_checkpoint.pt",
                           warmup=0, name=f"rule {w}")
        finetune.append({"rule_weight": w, "score": float(s), "best_epoch": e, "history": h})
        print(f"rule weight {w}: score {s:.3f} (main: {base_score:.3f})\n")
    winner = max(finetune, key=lambda t: t["score"])
    RULE_WEIGHT = winner["rule_weight"]
    shutil.copy(OUT_DIR / ("main_best.pt" if RULE_WEIGHT == 0 else f"rule_{RULE_WEIGHT}_best.pt"), WEIGHTS_V3)
    v3 = lc.LayoutCVAE2().to(DEVICE)
    v3.load_state_dict(torch.load(WEIGHTS_V3, map_location=DEVICE))
    TRAIN_LOG.write_text(json.dumps({"beta": BETA, "free_bits": FREE_BITS, "best_epoch": best_epoch, "history": history,
                                     "finetune": finetune, "rule_weight": RULE_WEIGHT,
                                     "minutes": (time.time() - start) / 60}, indent=1, default=float))
    print(f"v3 = {'the main training' if RULE_WEIGHT == 0 else f'the fine-tuning with rule weight {RULE_WEIGHT}'}, "
          f"saved to {WEIGHTS_V3} ({(time.time() - start) / 60:.0f} minutes in total)")
pd.DataFrame([{k: v for k, v in t.items() if k != "history"} for t in finetune]).round(3)

In [ ]:
h = pd.DataFrame(history)
checks = h.dropna(subset=["usable"])
fig, axes = plt.subplots(1, 4, figsize=(20, 3.8))
axes[0].plot(h["epoch"], h["val_bce"], label="validation BCE"); axes[0].plot(h["epoch"], h["val_dice"], label="validation Dice")
axes[0].legend(); axes[0].set_title("rebuilding error (lower = better)")
axes[1].plot(h["epoch"], h["val_kl"]); axes[1].set_title("KL (information in the random numbers)")
axes[2].plot(checks["epoch"], checks["bedrooms_right"], label="bedrooms right")
axes[2].plot(checks["epoch"], checks["different"], label="options differ")
axes[2].legend(); axes[2].set_ylim(0, 1); axes[2].set_title("raw pictures")
axes[3].plot(checks["epoch"], checks["usable"], marker="o", label="v3")
axes[3].axhline(V2_SELECT["usable"], color="#888780", ls="--", label="v2")
axes[3].axvline(best_epoch, color="#e34948", ls=":"); axes[3].legend()
axes[3].set_title(f"usable after step 3 (best epoch {best_epoch})")
for ax in axes:
    ax.set_xlabel("epoch")
plt.tight_layout()
save_fig("80_v3_training")
plt.show()

# 5. v1, v2 and v3 on the test plans

The test plans were never used: not for training, not for choosing an epoch. **40 test requests × 30 pictures**, through step 3 with the latest rules, the same random numbers for the three models. And the raw pictures of 300 test requests.

In [ ]:
v1 = lc.LayoutCVAE(latent_dim=config_v1["latent_dim"]).to(DEVICE)
v1.load_state_dict(torch.load(MODEL_V1_DIR / config_v1["weights"], map_location=DEVICE))
models = {"v1": v1, "v2": v2, "v3": v3}
raw_rows = np.random.default_rng(3).choice(test_rows, size=300, replace=False)
test_step3_rows = np.random.default_rng(4).choice(test_rows, size=40, replace=False)
summary, reasons_all = {}, {}
for name, m in models.items():
    start = time.time()
    raw = raw_score(m, raw_rows)
    s3 = step3_score(m, test_step3_rows, n=30, seed=5000)
    summary[name] = {"usable pictures (of 30)": 30 * s3["usable"], "requests with 5 options": s3["five_options"],
                     "bedrooms right (raw)": raw["bedrooms_right"], "balcony right (raw)": raw["balcony_right"],
                     "options differ": raw["different"]}
    reasons_all[name] = pd.Series(s3["reasons"]) / sum(s3["reasons"].values()) * 100
    print(f"{name}: {time.time() - start:.0f} s")
summary = pd.DataFrame(summary).T
display(summary.round(2))
print("What happened to the pictures (% of all):")
display(pd.DataFrame(reasons_all).fillna(0).round(1).sort_values("v3", ascending=False))
ok = summary[summary["options differ"] >= 0.3]                     # the 5 options must really be different
BEST = ok.sort_values(["usable pictures (of 30)", "requests with 5 options"], ascending=False).index[0]
print(f"\nBest for the app: {BEST}")

# 6. See it: the same 4 test requests, v1, v2 and v3

For each request: the real plan, then 5 options of each model (the best 5 of 30 after step 3).

In [ ]:
from matplotlib.patches import Patch

LEGEND = [Patch(facecolor=lc.COLORS[t], edgecolor="#3d3d3a", label=t) for t in lc.ROOMS + ["wall", "window", "door", "entrance"]]
demo_rows = test_step3_rows[:4]
fig, axes = plt.subplots(12, 6, figsize=(24, 48))
for i, r in enumerate(demo_rows):
    b, y = int(lib.loc[r, "bedrooms"]), int(lib.loc[r, "balcony"])
    for j, (name, m) in enumerate(models.items()):
        row = axes[3 * i + j]
        if j == 0:
            row[0].imshow(lc.to_color(maps_u8[r], sil_u8[r]))
            row[0].set_title(f"REAL · {b} bed · balcony {'yes' if y else 'no'}", fontsize=12)
            row[0].set_xticks([]); row[0].set_yticks([])
        else:
            row[0].set_axis_off()
        maps = lc.generate(m, sil_u8[r], conds[r], n=30, seed=i, device=DEVICE)
        options, _ = lc.design_from_maps(maps, sil_u8[r], b, y, lib.loc[r, "m_per_px"], rules)
        for c in range(5):
            if c < len(options):
                lc.draw_layout(options[c], row[c + 1], f"{name} · option {c + 1} · score {options[c]['score']:.2f}")
            else:
                row[c + 1].set_axis_off()
                row[c + 1].text(0.5, 0.5, f"{name}: no option {c + 1}", ha="center", va="center", transform=row[c + 1].transAxes)
fig.legend(handles=LEGEND, loc="upper center", ncol=len(LEGEND), frameon=False, bbox_to_anchor=(0.5, 1.005))
plt.tight_layout()
save_fig("81_v1_v2_v3")
plt.show()

# 7. Save v3 for the server

v3 uses v2's network, so the config says `"architecture": "v2"` and the server needs no change. The files go to `OUT_DIR` and, on Colab, are downloaded as one zip at the end. Put them in **`01_autoencoder/models/layout_generator_v3/`** on your computer. If section 5 says v3 is the best, in `02_server`: `python publish.py --model layout_generator_v3`, try it with `python app.py`, then push.

In [ ]:
def check(name):
    s = summary.loc[name]
    return {"step 3 on test: usable pictures of 30": float(s["usable pictures (of 30)"]),
            "step 3 on test: requests with 5 options": float(s["requests with 5 options"]),
            "invented: bedrooms right": float(s["bedrooms right (raw)"]),
            "invented: balcony as asked": float(s["balcony right (raw)"]),
            "options differ (share of pixels)": float(s["options differ"])}


config_v3 = {**config_v1,
             "model": "LayoutCVAE2 (v2's network, FiLM), trained as v3: epoch chosen by step 3, heavier small rooms, rule loss",
             "architecture": "v2", "version": "v3", "weights": WEIGHTS_V3.name, "latent_dim": v3.latent_dim,
             "beta": BETA, "free_bits": FREE_BITS, "dice_weights": DICE_WEIGHTS.flatten().tolist(), "rule_weight": RULE_WEIGHT,
             "epochs_trained": len(history), "best_epoch": best_epoch,
             "check_on_test": {name: check(name) for name in models}, "best_for_the_app": BEST}
for key in ["check_on_validation", "v1_on_the_same_requests", "v2_is_better", "aux_weight"]:
    config_v3.pop(key, None)
(OUT_DIR / "layout_cvae_config.json").write_text(json.dumps(config_v3, indent=2))
shutil.copy(MODEL_V1_DIR / "room_rules.json", OUT_DIR / "room_rules.json")


class Decoder(nn.Module):                         # check that the ONNX export works (publish.py makes it again)
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, z, s, c):
        return self.m.decode(z, s, c)


cpu_model = lc.LayoutCVAE2(); cpu_model.load_state_dict(torch.load(WEIGHTS_V3, map_location="cpu")); cpu_model.eval()
z, s, c = torch.randn(3, cpu_model.latent_dim), (torch.rand(3, 1, 128, 128) > 0.3).float(), torch.rand(3, 7)
options = dict(input_names=["z", "s", "c"], output_names=["maps"], opset_version=17,
               dynamic_axes={"z": {0: "n"}, "s": {0: "n"}, "c": {0: "n"}, "maps": {0: "n"}})
onnx_path = OUT_DIR / "layout_decoder.onnx"
try:
    torch.onnx.export(Decoder(cpu_model), (z, s, c), str(onnx_path), dynamo=False, **options)
except TypeError:
    torch.onnx.export(Decoder(cpu_model), (z, s, c), str(onnx_path), **options)
import onnxruntime as ort
diff = abs(ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"]).run(
    None, {"z": z.numpy(), "s": s.numpy(), "c": c.numpy()})[0] - cpu_model.decode(z, s, c).detach().numpy()).max()
print(f"ONNX check: largest difference {diff:.1e}")
for f in sorted(OUT_DIR.iterdir()):
    print(f"{f.name:34s} {f.stat().st_size / 1e6:7.2f} MB")

In [ ]:
if IN_COLAB:                    # v3 in one zip, downloaded to your computer (without the checkpoints: not needed there)
    pack = Path("/content/v3_pack")
    shutil.rmtree(pack, ignore_errors=True)
    pack.mkdir()
    for f in OUT_DIR.iterdir():
        if f.is_file() and "checkpoint" not in f.name and not (f.name.endswith("_best.pt")):
            shutil.copy(f, pack / f.name)
    archive = shutil.make_archive("/content/v3_results", "zip", pack)
    print(f"{archive} ({Path(archive).stat().st_size / 1e6:.0f} MB)")
    files.download(archive)

# 8. Conclusions

*(written after the run, from the numbers in sections 4 and 5)*